# Project 10: Suicidal Ideation Detection
**Team No.:** 25  
**Team Members:** Rohit Hazra; Tanmay Mohanty; Shubhranshu Sahoo; Diptiranjan Behera  
**Task:** Classification  
**Proposed Hybrid:** DeBERTa + Contextual Interaction GAT  
**Dataset:** [Annotated Reddit suicidal ideation dataset](https://www.kaggle.com/datasets/rvarun11/suicidal-ideation-reddit-dataset)

This executable Colab notebook discovers the downloaded schema defensively, prevents split leakage, trains the complete proposed model, reloads the best validation checkpoints, evaluates the test set once, and writes reproducible artifacts.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers tqdm tabulate

In [ ]:
import os, json, random, shutil, glob, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print("Device:", DEVICE)

### CONFIG

In [ ]:
CONFIG = {
    "project_no": "10",
    "project_name": "Suicidal Ideation Detection",
    "team_no": "25",
    "task_type": "classification",
    "kaggle_dataset_slug": "rvarun11/suicidal-ideation-reddit-dataset",
    "target_candidates": ["label", "class", "target", "suicidal", "is_suicide"],
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "model_name": "microsoft/deberta-v3-small",
    "max_length": 160,
    "epochs": 8,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "early_stop_patience": 3,
    "use_amp": True,
    "data_raw_dir": "data/10/raw",
    "data_processed_dir": "data/10/processed",
    "figures_dir": "data/10/figures",
    "results_dir": "data/10/results",
    "reports_dir": "data/10/reports",
}
# Single source of truth for the checkpoint path - training and every eval/reload cell below
# read/write through this key rather than reconstructing the path (or hardcoding it), which is
# what previously let the save and load paths quietly drift apart.
CONFIG["checkpoint_path"] = os.path.join(CONFIG["results_dir"], "best_hybrid.pt")
CONFIG["metrics_path"] = os.path.join(CONFIG["results_dir"], "metrics.json")

# Every directory any cell in this notebook writes to is created here, upfront, in one loop.
for key in ["data_raw_dir", "data_processed_dir", "figures_dir", "results_dir", "reports_dir"]:
    os.makedirs(CONFIG[key], exist_ok=True)
CONFIG

## 1. Dataset Download

In [ ]:
import kagglehub
cache_path = kagglehub.dataset_download(CONFIG["kaggle_dataset_slug"])
source = pathlib.Path(cache_path)
destination = pathlib.Path(CONFIG["data_raw_dir"])
for item in source.rglob("*"):
    if item.is_file():
        relative = item.relative_to(source)
        output = destination / relative
        output.parent.mkdir(parents=True, exist_ok=True)
        if not output.exists() or output.stat().st_size != item.stat().st_size:
            shutil.copy2(item, output)

raw_files = [p for p in destination.rglob("*") if p.is_file()]
assert raw_files, "Dataset download produced no files."
assert all(p.stat().st_size > 0 for p in raw_files), "A downloaded file is empty."
print(f"Discovered {len(raw_files)} non-empty files")
print(*[str(p) for p in raw_files[:20]], sep="\n")

## 2. Load Raw Data

In [ ]:
csvs = [p for p in raw_files if p.suffix.lower() == ".csv"]
assert csvs, "No CSV files found among the downloaded files."

frames = []
for path in csvs:
    try:
        f = pd.read_csv(path)
        text_candidates = [c for c in f.columns if any(k in c.lower() for k in ["text", "post", "content", "sentence"])]
        label_candidates = [c for c in CONFIG["target_candidates"] if c in f.columns]
        if text_candidates and label_candidates:
            frames.append((path, f, text_candidates[0], label_candidates[0]))
    except Exception as exc:
        print("Skipped", path, exc)

assert frames, "No CSV with a defensible text column and target column was found."
RAW_FILE, df, text_col, target = max(frames, key=lambda z: len(z[1]))
df = df[[text_col, target]].dropna().drop_duplicates().reset_index(drop=True)
print(RAW_FILE, df.shape, "text_col=", text_col, "target=", target)
df.head()

In [ ]:
# [TARGET SANITY CHECK] - right after the label column is identified. Hard-fails on a
# degenerate (single-class) target; soft-warns on tiny/imbalanced data.
print("Target distribution (counts):")
print(df[target].value_counts())
print("Target distribution (normalized):")
print(df[target].value_counts(normalize=True))
assert df[target].nunique() > 1, (
    f"DEGENERATE TARGET: only {df[target].nunique()} unique value(s) found - "
    f"{df[target].value_counts().to_dict()}. Check upstream row-limiting/sorting/filtering logic before proceeding."
)
minority_frac = df[target].value_counts(normalize=True).min()
if minority_frac < 0.01 or len(df) < 100:
    print(f"[DATA QUALITY WARNING] rows={len(df)}, minority class fraction={minority_frac:.4f} - "
          "check upstream filtering/sampling.")

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
word_counts = df[text_col].astype(str).str.split().str.len()

plt.figure(figsize=(7, 4))
sns.histplot(word_counts, bins=40)
plt.xlabel("Words per post")
plt.title("Post length distribution")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_text_length.png"), dpi=150)
plt.show()

class_balance = df[target].value_counts(normalize=True).to_dict()
memo = f"""# Data Quality Memo - Project 10: Suicidal Ideation Detection

## Dataset
- Source file: {RAW_FILE}
- Rows after null removal and de-duplication: {len(df)}
- Text column: {text_col}; target column: {target}
- Mean words per post: {word_counts.mean():.1f} (min={int(word_counts.min())}, max={int(word_counts.max())})
- Class balance: {class_balance}

## Known limitations
- Labels come from a community-annotated Reddit dataset, not a clinical diagnosis; they reflect
  annotator judgement about post content, not verified suicidality.
- The stratified split happens before tokenization, and duplicate posts are removed up front,
  so the same post cannot appear in more than one split.
- This is research code for a coursework project, not a clinical decision system, and must not
  be used to make real mental-health decisions about any individual.
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w", encoding="utf-8") as f:
    f.write(memo)
print(memo)

## 4. Preprocessing & Feature Engineering

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME = CONFIG["model_name"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
le = LabelEncoder()
y = le.fit_transform(df[target].astype(str))
n_classes = len(le.classes_)
print("Classes:", le.classes_.tolist(), "| n_classes:", n_classes)

## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]
idx = np.arange(len(df))
tr, rest = train_test_split(idx, train_size=ratios["train"], stratify=y, random_state=SEED)
rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
va, te = train_test_split(rest, train_size=rel_val, stratify=y[rest], random_state=SEED)

assert not (set(tr) & set(va) or set(tr) & set(te) or set(va) & set(te)), "Split leakage detected"
for split_name, split_idx in [("train", tr), ("val", va), ("test", te)]:
    assert len(np.unique(y[split_idx])) == n_classes, f"{split_name} split lost a class"

manifest = {
    "train": len(tr), "val": len(va), "test": len(te), "classes": le.classes_.tolist(),
    "train_class_balance": pd.Series(y[tr]).value_counts(normalize=True).sort_index().to_dict(),
    "val_class_balance": pd.Series(y[va]).value_counts(normalize=True).sort_index().to_dict(),
    "test_class_balance": pd.Series(y[te]).value_counts(normalize=True).sort_index().to_dict(),
}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)
manifest

## 6. PyTorch Dataset & DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, indices):
        self.rows = df.iloc[indices].reset_index(drop=True)
        self.labels = y[indices]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row = self.rows.iloc[i]
        enc = tokenizer(
            str(row[text_col]), max_length=CONFIG["max_length"], truncation=True,
            padding="max_length", return_tensors="pt",
        )
        return enc["input_ids"][0], enc["attention_mask"][0], torch.tensor(self.labels[i], dtype=torch.long)

train_loader = DataLoader(TextDataset(tr), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(TextDataset(va), batch_size=CONFIG["batch_size"])
test_loader = DataLoader(TextDataset(te), batch_size=CONFIG["batch_size"])

ids, mask, yb = next(iter(train_loader))
print("input_ids:", ids.shape, "attention_mask:", mask.shape, "target:", yb.shape)

## 7. Proposed Model Definition — DeBERTa + Contextual Interaction GAT

In [ ]:
class ContextualInteractionGAT(nn.Module):
    """Single-head graph-attention layer over sentence-boundary token nodes extracted from the
    DeBERTa hidden states; the adjacency is the dense token-node mask (a fully connected graph
    restricted to valid, non-padding node positions)."""
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)

    def forward(self, nodes, node_mask):
        scores = self.q(nodes) @ self.k(nodes).transpose(1, 2) / nodes.shape[-1] ** 0.5
        adjacency = node_mask[:, None, :] & node_mask[:, :, None]
        scores = scores.masked_fill(~adjacency, -10000.0)
        attention = torch.softmax(scores, dim=-1)
        return attention @ self.v(nodes), attention


class DeBERTaInteractionGAT(nn.Module):
    def __init__(self, k, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        d = self.encoder.config.hidden_size
        self.gat = ContextualInteractionGAT(d)
        self.head = nn.Linear(2 * d, k)
        self.last_attention = None
        self._period_id = None  # resolved lazily against the live tokenizer

    def forward(self, ids, mask):
        tokens = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state.float()
        if self._period_id is None:
            self._period_id = tokenizer.convert_tokens_to_ids(".")
        boundaries = (
            (ids == self._period_id) |
            (torch.arange(ids.shape[1], device=ids.device)[None, :] % 32 == 0)
        ) & mask.bool()
        boundaries[:, 0] = True  # guarantee at least one node (CLS position) per sample
        nodes = tokens * boundaries.unsqueeze(-1)
        graph, att = self.gat(nodes, boundaries)
        graph = graph * boundaries.unsqueeze(-1)
        self.last_attention = att * boundaries.unsqueeze(1) * boundaries.unsqueeze(2)
        denom = boundaries.sum(1, keepdim=True).clamp_min(1)
        pooled_graph = graph.sum(1) / denom
        return self.head(torch.cat([tokens[:, 0], pooled_graph], dim=1))

### Architecture Verification

In [ ]:
hybrid = DeBERTaInteractionGAT(n_classes, MODEL_NAME).to(DEVICE)
total = sum(p.numel() for p in hybrid.parameters())
trainable = sum(p.numel() for p in hybrid.parameters() if p.requires_grad)
print(f"hybrid: total={total:,} trainable={trainable:,} device={next(hybrid.parameters()).device}")

## 8. Training Loop

In [ ]:
def train_model(model, checkpoint_path, epochs, patience, lr):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    use_amp = bool(CONFIG.get("use_amp", True)) and DEVICE.type == "cuda"
    try:
        amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best = float("inf")
    wait = 0
    hist = {"train_loss": [], "val_loss": []}
    epoch_bar = tqdm(range(epochs), desc="Training", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        total, n = 0.0, 0
        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False, unit="batch")
        for ids, mask, yb in batch_bar:
            ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(ids, mask), yb)
            if amp_scaler.is_enabled():
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                amp_scaler.step(opt)
                amp_scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            total += loss.item() * len(ids)
            n += len(ids)
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")
        trn = total / max(n, 1)

        model.eval()
        val_total, val_n = 0.0, 0
        with torch.no_grad():
            for ids, mask, yb in val_loader:
                ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    val_loss = F.cross_entropy(model(ids, mask), yb)
                val_total += val_loss.item() * len(ids)
                val_n += len(ids)
        va = val_total / max(val_n, 1)

        hist["train_loss"].append(trn)
        hist["val_loss"].append(va)
        epoch_bar.set_postfix(train_loss=f"{trn:.4f}", val_loss=f"{va:.4f}")

        if va < best:
            best = va
            wait = 0
            torch.save(model.state_dict(), checkpoint_path)
            # Verification step: catch a silent save failure immediately instead of a
            # confusing FileNotFoundError later, at load time. This is the exact class of bug
            # ("checkpoint was never actually saved") this assertion exists to prevent.
            assert os.path.exists(checkpoint_path), f"Checkpoint save failed: {checkpoint_path}"
        else:
            wait += 1
            if wait >= patience:
                epoch_bar.write(f"Early stopping at epoch {epoch + 1}")
                break

    assert os.path.exists(checkpoint_path), f"No checkpoint was ever saved at {checkpoint_path}"
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    return model, hist


hybrid, hybrid_history = train_model(
    hybrid, CONFIG["checkpoint_path"], epochs=CONFIG["epochs"],
    patience=CONFIG["early_stop_patience"], lr=CONFIG["learning_rate"],
)

## 9. Evaluation Metrics

In [ ]:
def get_predictions(model, loader):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for ids, mask, yb in loader:
            out = torch.softmax(model(ids.to(DEVICE), mask.to(DEVICE)), dim=1)
            probs.append(out.cpu().numpy())
            targets.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(targets)


def evaluate_classification(probs, targets):
    pred = probs.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, pred, average="macro", zero_division=0)
    out = {"accuracy": accuracy_score(targets, pred), "precision_macro": precision,
           "recall_macro": recall, "f1_macro": f1}
    if probs.shape[1] == 2 and len(np.unique(targets)) == 2:
        out["roc_auc"] = roc_auc_score(targets, probs[:, 1])
        out["pr_auc"] = average_precision_score(targets, probs[:, 1])
    return out


hybrid_prob, test_y2 = get_predictions(hybrid, test_loader)
test_y = y[te]
assert np.array_equal(test_y, test_y2), "Test-set label ordering mismatch between split and DataLoader."
results = {"hybrid": evaluate_classification(hybrid_prob, test_y)}
with open(CONFIG["metrics_path"], "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Reload-and-verify step: load the checkpoint back into a fresh model instance and confirm it
# reproduces the metrics above. This specifically guards against the checkpoint-save silently
# failing while training appears to finish normally.
verify_model = DeBERTaInteractionGAT(n_classes, MODEL_NAME).to(DEVICE)
verify_model.load_state_dict(torch.load(CONFIG["checkpoint_path"], map_location=DEVICE, weights_only=True))
verify_prob, verify_y = get_predictions(verify_model, test_loader)
verify_metrics = evaluate_classification(verify_prob, verify_y)
for k in results["hybrid"]:
    assert abs(results["hybrid"][k] - verify_metrics[k]) < 1e-6, (
        f"Reload mismatch on {k}: trained={results['hybrid'][k]} reloaded={verify_metrics[k]}"
    )
print("Reload-and-verify PASSED:", verify_metrics)
del verify_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history["train_loss"], label="train")
plt.plot(hybrid_history["val_loss"], label="val")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.title("Hybrid model training / validation loss")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig01_loss_curves.png"), dpi=150)
plt.show()

In [ ]:
pred = hybrid_prob.argmax(1)
cm = confusion_matrix(test_y, pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
if hybrid_prob.shape[1] == 2 and len(np.unique(test_y)) == 2:
    fpr, tpr, _ = roc_curve(test_y, hybrid_prob[:, 1])
    precision, recall, _ = precision_recall_curve(test_y, hybrid_prob[:, 1])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate"); axes[0].set_title("ROC Curve")
    axes[1].plot(recall, precision)
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=150)
    plt.show()
else:
    per_class_recall = np.diag(cm) / np.maximum(cm.sum(1), 1)
    plt.figure(figsize=(6, 4))
    plt.bar(range(n_classes), per_class_recall)
    plt.xticks(range(n_classes), le.classes_)
    plt.ylabel("Per-class recall")
    plt.title("Per-class recall (test set)")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_per_class_recall.png"), dpi=150)
    plt.show()

### Explainability — Contextual Interaction GAT Attention

In [ ]:
# Visualize the GAT attention matrix over sentence-boundary token nodes for one test sample.
hybrid.eval()
ids, mask, yb = next(iter(test_loader))
with torch.no_grad():
    hybrid(ids.to(DEVICE), mask.to(DEVICE))
att = hybrid.last_attention[0].detach().cpu().numpy()
k = min(40, att.shape[0])

plt.figure(figsize=(7, 6))
sns.heatmap(att[:k, :k], cmap="viridis")
plt.title("Contextual Interaction GAT attention (first %d nodes, one test sample)" % k)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=150)
plt.show()

### Error Analysis

In [ ]:
errors = pred != test_y
n_err = int(errors.sum())
print(f"Misclassified test rows: {n_err:,} / {len(test_y):,} ({n_err / max(len(test_y), 1):.2%})")

plt.figure(figsize=(7, 4))
sns.histplot(hybrid_prob.max(1)[errors], bins=20)
plt.xlabel("Predicted-class confidence on misclassified samples")
plt.title("Confidence distribution of held-out errors")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=150)
plt.show()